In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

sns.set_theme(style="whitegrid")

df = pd.read_csv("data/titanic.csv")
print("Datos cargados correctamente")

## Capítulo 1: El primer vistazo a la tragedia

In [ ]:
# 1. Total de pasajeros
total_pasajeros = len(df)
print(f"Total de pasajeros: {total_pasajeros}")

# 2. Sobrevivientes y fallecidos
sobrevivientes = df["Survived"].sum()
fallecidos = total_pasajeros - sobrevivientes
print(f"Sobrevivieron: {sobrevivientes}")
print(f"Fallecieron: {fallecidos}")

# 3. Tasa de supervivencia global
tasa_global = (sobrevivientes / total_pasajeros) * 100
print(f"\nTasa de supervivencia global: {tasa_global:.1f}%")

# 4. Datos faltantes
print("\nDatos faltantes por columna:")
print(df.isna().sum()[df.isna().sum() > 0])

## Capítulo 2: La hipótesis de la clase social

In [ ]:
# 1. Tasa de supervivencia por clase
supervivencia_clase = df.groupby("Pclass")["Survived"].agg(["mean", "count"])
supervivencia_clase["tasa_%"] = (supervivencia_clase["mean"] * 100).round(1)
print("Supervivencia por clase:")
print(supervivencia_clase[["count", "tasa_%"]])

# 2 y 3. Grafico
plt.figure(figsize=(8, 5))
sns.barplot(data=df, x="Pclass", y="Survived", estimator="mean", errorbar=None)
plt.title("Tasa de supervivencia por clase", fontsize=14, fontweight="bold")
plt.xlabel("Clase (1=Primera, 2=Segunda, 3=Tercera)")
plt.ylabel("Tasa de supervivencia")
plt.ylim(0, 1)
for i, v in enumerate(supervivencia_clase["mean"]):
    plt.text(i, v + 0.02, f"{v*100:.1f}%", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

# 4. Conclusion
clase_max = supervivencia_clase["mean"].idxmax()
print(f"\nCONCLUSION: La clase {clase_max} tuvo la mayor supervivencia.")
print("Los rumores ERAN CIERTOS: la clase social influyó significativamente.")
print(f"1ra clase: {supervivencia_clase.loc[1, 'tasa_%']}% supervivencia")
print(f"3ra clase: {supervivencia_clase.loc[3, 'tasa_%']}% supervivencia")

## Capítulo 3: Mujeres y niños primero

In [ ]:
# 1. Supervivencia por sexo
print("Supervivencia por sexo:")
supervivencia_sexo = df.groupby("Sex")["Survived"].mean() * 100
print(supervivencia_sexo.round(1))

# 2. Crear columna age_group
df["age_group"] = df["Age"].apply(lambda x: "child" if pd.notna(x) and x < 18 else "adult")

# 3. Supervivencia por grupo de edad
print("\nSupervivencia por grupo de edad:")
supervivencia_edad = df.groupby("age_group")["Survived"].mean() * 100
print(supervivencia_edad.round(1))

# 4. Grafico completo
plt.figure(figsize=(10, 6))
supervivencia_completa = df.groupby(["Pclass", "Sex"])["Survived"].mean().reset_index()
sns.barplot(data=supervivencia_completa, x="Pclass", y="Survived", hue="Sex")
plt.title("Supervivencia por clase y sexo", fontsize=14, fontweight="bold")
plt.xlabel("Clase")
plt.ylabel("Tasa de supervivencia")
plt.ylim(0, 1)
plt.legend(title="Sexo")
plt.tight_layout()
plt.show()

# 5. Conclusion
print("\nCONCLUSION:")
print(f"- Mujeres: {supervivencia_sexo['female']:.1f}% supervivencia")
print(f"- Hombres: {supervivencia_sexo['male']:.1f}% supervivencia")
print(f"- Ninos: {supervivencia_edad.get('child', 0):.1f}% supervivencia")
print("\nSI se cumplió 'mujeres y niños primero', especialmente en 1ra y 2da clase.")

## Capítulo 4: El factor precio del billete

In [ ]:
# 1. Precio promedio por supervivencia
precio_por_supervivencia = df.groupby("Survived")["Fare"].mean()
print("Precio promedio del billete:")
print(f"No sobrevivientes: ${precio_por_supervivencia[0]:.2f}")
print(f"Sobrevivientes: ${precio_por_supervivencia[1]:.2f}")

# 2. Crear categorias de precio
df["fare_category"] = pd.cut(
    df["Fare"],
    bins=[0, 20, 50, float("inf")],
    labels=["low", "medium", "high"]
)

# 3. Supervivencia por categoria
print("\nSupervivencia por categoria de precio:")
supervivencia_precio = df.groupby("fare_category")["Survived"].mean() * 100
print(supervivencia_precio.round(1))

# 4. Boxplot
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x="Survived", y="Fare")
plt.title("Distribucion del precio por supervivencia", fontsize=14, fontweight="bold")
plt.xlabel("Sobrevivio (0=No, 1=Si)")
plt.ylabel("Precio del billete ($)")
plt.tight_layout()
plt.show()

# 5. Conclusion
diferencia = precio_por_supervivencia[1] - precio_por_supervivencia[0]
print(f"\nCONCLUSION: Los sobrevivientes pagaron ${diferencia:.2f} más en promedio.")
print("SI, pagar más aumentó las probabilidades (correlación con clase social).")

## Capítulo 5: El perfil del superviviente ideal

In [ ]:
# 1. Combinación con MAYOR supervivencia
perfil_combinado = df.groupby(["Pclass", "Sex"])["Survived"].agg(["mean", "count"]).reset_index()
perfil_combinado = perfil_combinado[perfil_combinado["count"] >= 10]  # minimo 10 casos
mejor_perfil = perfil_combinado.loc[perfil_combinado["mean"].idxmax()]
peor_perfil = perfil_combinado.loc[perfil_combinado["mean"].idxmin()]

print("PERFIL CON MAYOR SUPERVIVENCIA:")
print(f"Clase: {int(mejor_perfil['Pclass'])}, Sexo: {mejor_perfil['Sex']}")
print(f"Tasa: {mejor_perfil['mean']*100:.1f}%\n")

print("PERFIL CON MENOR SUPERVIVENCIA:")
print(f"Clase: {int(peor_perfil['Pclass'])}, Sexo: {peor_perfil['Sex']}")
print(f"Tasa: {peor_perfil['mean']*100:.1f}%\n")

# 3. Edad promedio
edad_por_supervivencia = df.groupby("Survived")["Age"].mean()
print(f"Edad promedio no sobrevivientes: {edad_por_supervivencia[0]:.1f} años")
print(f"Edad promedio sobrevivientes: {edad_por_supervivencia[1]:.1f} años\n")

# 4. Perfil superviviente ideal
supervivientes = df[df["Survived"] == 1]
print("=" * 50)
print("PERFIL DEL SUPERVIVIENTE IDEAL:")
print("=" * 50)
print(f"- Clase: 1ra clase")
print(f"- Sexo: Mujer")
print(f"- Edad: 20-40 años")
print(f"- Precio billete: > $50")
print(f"- Probabilidad estimada: ~{mejor_perfil['mean']*100:.0f}%")

# 5. Perfil víctima más probable
print("\n" + "=" * 50)
print("PERFIL DE VÍCTIMA MÁS PROBABLE:")
print("=" * 50)
print(f"- Clase: 3ra clase")
print(f"- Sexo: Hombre")
print(f"- Edad: 20-40 años")
print(f"- Precio billete: < $20")
print(f"- Probabilidad de no sobrevivir: ~{(1-peor_perfil['mean'])*100:.0f}%")

In [ ]:
# 6. BONUS: Visualización creativa de perfiles
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Grafico 1: Heatmap de supervivencia
pivot_survival = df.pivot_table(values="Survived", index="Pclass", columns="Sex", aggfunc="mean")
sns.heatmap(pivot_survival, annot=True, fmt=".2f", cmap="RdYlGn", ax=axes[0], vmin=0, vmax=1)
axes[0].set_title("Mapa de calor: Tasa de supervivencia", fontweight="bold")
axes[0].set_xlabel("Sexo")
axes[0].set_ylabel("Clase")

# Grafico 2: Distribucion de edad por supervivencia
df_clean = df.dropna(subset=["Age"])
sns.violinplot(data=df_clean, x="Survived", y="Age", hue="Sex", split=True, ax=axes[1])
axes[1].set_title("Distribucion de edad por supervivencia y sexo", fontweight="bold")
axes[1].set_xlabel("Sobrevivio (0=No, 1=Si)")
axes[1].set_ylabel("Edad")

plt.tight_layout()
plt.show()